# Milestone 1: Data Profiling

Profil awal dataset besar Indonesia menggunakan Polars LazyFrame dan DuckDB. Isi sumber serta deskripsi dataset pada `data/README.md` sebelum menjalankan notebook ini.

In [1]:
import duckdb, polars as pl, os, glob

DATA_PATH = "../data/raw/domain_dfk.jsonl"
RAW_PQ = "../data/processed/data.parquet"
os.makedirs("../data/processed", exist_ok=True)

if not os.path.exists(RAW_PQ):
    duckdb.sql(f"""
    COPY (
      SELECT row_number() OVER () AS doc_id, text
      FROM read_json('{DATA_PATH}', format='newline_delimited',
                     columns={{'text': 'VARCHAR'}}, ignore_errors=true)
    ) TO '{RAW_PQ}' (FORMAT PARQUET, COMPRESSION ZSTD)
    """)
print("Parquet siap:", RAW_PQ)

Parquet siap: ../data/processed/data.parquet


In [2]:
raw_mb = os.path.getsize(DATA_PATH) / 1e6
pq_mb = os.path.getsize(RAW_PQ) / 1e6
n = pl.scan_parquet(RAW_PQ).select(pl.len()).collect().item()
print(f"JSONL: {raw_mb:,.0f} MB | Parquet: {pq_mb:,.0f} MB | Baris: {n:,}")
print("Memenuhi syarat (>=500 MB atau >1 juta baris):", raw_mb >= 500 or n > 1_000_000)

JSONL: 654 MB | Parquet: 213 MB | Baris: 310,930
Memenuhi syarat (>=500 MB atau >1 juta baris): True


## Profil Polars

`scan_parquet` hanya membangun rencana query; file baru dibaca ketika `collect()` dipanggil.

In [3]:
lf = pl.scan_parquet(RAW_PQ)
print(lf.collect_schema())
lf.head(3).collect().with_columns(pl.col("text").str.slice(0, 200))

Schema({'doc_id': Int64, 'text': String})


doc_id,text
i64,str
1,"""Polisi mengapresiasi aksi dama…"
2,"""Bupati Bandung Dadang Supriatn…"
3,"""Komisi I dan Komisi III DPRD J…"


In [4]:
prof = lf.select(
    pl.col("text").null_count().alias("null"),
    (pl.col("text").str.strip_chars() == "").sum().alias("kosong"),
    pl.col("text").str.len_chars().min().alias("min_char"),
    pl.col("text").str.len_chars().median().alias("median_char"),
    pl.col("text").str.len_chars().quantile(0.99).alias("p99_char"),
    pl.col("text").str.len_chars().max().alias("max_char"),
    pl.col("text").str.split(" ").list.len().mean().alias("rata2_kata"),
    pl.col("text").n_unique().alias("teks_unik"),
).collect()
print(prof)
print(f"Duplikat teks: {n - prof['teks_unik'][0]:,}")

shape: (1, 8)
┌──────┬────────┬──────────┬─────────────┬──────────┬──────────┬────────────┬───────────┐
│ null ┆ kosong ┆ min_char ┆ median_char ┆ p99_char ┆ max_char ┆ rata2_kata ┆ teks_unik │
│ ---  ┆ ---    ┆ ---      ┆ ---         ┆ ---      ┆ ---      ┆ ---        ┆ ---       │
│ u32  ┆ u32    ┆ u32      ┆ f64         ┆ f64      ┆ u32      ┆ f64        ┆ u32       │
╞══════╪════════╪══════════╪═════════════╪══════════╪══════════╪════════════╪═══════════╡
│ 0    ┆ 0      ┆ 23       ┆ 1740.0      ┆ 7894.0   ┆ 149444   ┆ 287.398736 ┆ 310930    │
└──────┴────────┴──────────┴─────────────┴──────────┴──────────┴────────────┴───────────┘
Duplikat teks: 0


## Profil DuckDB

DuckDB membaca file Parquet langsung dari disk dan menghasilkan ringkasan statistik setiap kolom.

In [5]:
duckdb.sql(f"SUMMARIZE SELECT * FROM '{RAW_PQ}'").show(max_width=250)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬─────────────┬──────────────────────────────────────────────────────────┬─────────────────────────────────────────────────────────┬───────────────┬──────────┬───────────────────┬─────────┬─────────┬─────────┬────────┬─────────────────┐
│ column_name │ column_type │                           min                            │                           max                           │ approx_unique │   avg    │        std        │   q25   │   q50   │   q75   │ count  │ null_percentage │
│   varchar   │   varchar   │                         varchar                          │                         varchar                         │     int64     │ varchar  │      varchar      │ varchar │ varchar │ varchar │ int64  │  decimal(9,2)   │
├─────────────┼─────────────┼──────────────────────────────────────────────────────────┼─────────────────────────────────────────────────────────┼───────────────┼──────────┼───────────────────┼─────────┼─────────┼─────────┼────────┼───────────────

In [6]:
# Teks dengan spasi aneh di tengah kata (mis. "Bertem pat", "p impinan"), indikasi noise OCR/scraping
noise = lf.select(
    pl.col("text").str.contains(r"\b[a-z] [a-z]{1,2} [a-z]{2,}\b").mean().alias("rasio_spasi_aneh"),
    pl.col("text").str.contains(r"(?i)https?://").mean().alias("rasio_url"),
).collect()
print(noise)

shape: (1, 2)
┌──────────────────┬───────────┐
│ rasio_spasi_aneh ┆ rasio_url │
│ ---              ┆ ---       │
│ f64              ┆ f64       │
╞══════════════════╪═══════════╡
│ 0.001814         ┆ 0.000035  │
└──────────────────┴───────────┘


## Catatan Milestone 1

Tulis temuan awal di bawah ini: ukuran data, jumlah baris, tipe kolom, kolom dengan null terbanyak, serta anomali yang perlu ditangani pada Milestone 2.